<a href="https://colab.research.google.com/github/dashy0070/Cybersecurity_AI/blob/main/NIDS0_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NIDS Development

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("galaxyh/kdd-cup-1999-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'kdd-cup-1999-data' dataset.
Path to dataset files: /kaggle/input/kdd-cup-1999-data


In [ ]:
column_names = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files',
    'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count',
    'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate',
    'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label'
]

data_file = os.path.join(path, 'kddcup.data_10_percent.gz')
df = pd.read_csv(data_file, names=column_names)

print(f"Successfully loaded {os.path.basename(data_file)} into a DataFrame with {df.shape[0]} rows and {df.shape[1]} columns.")
print(df.head())

Successfully loaded kddcup.data_10_percent.gz into a DataFrame with 494021 rows and 42 columns.
   duration protocol_type  ... dst_host_srv_rerror_rate    label
0         0           tcp  ...                      0.0  normal.
1         0           tcp  ...                      0.0  normal.
2         0           tcp  ...                      0.0  normal.
3         0           tcp  ...                      0.0  normal.
4         0           tcp  ...                      0.0  normal.

[5 rows x 42 columns]


In [ ]:
import os

# The 'path' variable already holds '/kaggle/input/kdd-cup-1999-data'
# List all files and directories within the downloaded dataset path
file_list = os.listdir(path)

print(f"Files in '{path}':")
for file_name in file_list:
    print(file_name)

Files in '/kaggle/input/kdd-cup-1999-data':
kddcup.data.gz
kddcup.data
kddcup.testdata.unlabeled
training_attack_types
kddcup.data.corrected
kddcup.newtestdata_10_percent_unlabeled.gz
corrected
kddcup.testdata.unlabeled.gz
kddcup.testdata.unlabeled_10_percent
corrected.gz
kddcup.newtestdata_10_percent_unlabeled
kddcup.data_10_percent.gz
kddcup.names
kddcup.data_10_percent_corrected
typo-correction.txt
kddcup.data_10_percent
kddcup.testdata.unlabeled_10_percent.gz


Add a column called
'Attack_class'. If the attack is 'normal', attack_class is normal, else 'attack'

Reason: This helps to reformulate the problem as two stage problem or not.
First classify into attack and attack_class.

In [ ]:
import numpy as np
df['Attack_class'] = np.where(df['label'] == 'normal.', 'normal', 'attack')
attack_class_distribution = df['Attack_class'].value_counts()
print("Distribution of 'Attack_class' column:")
print(attack_class_distribution)

Distribution of 'Attack_class' column:
Attack_class
attack    396743
normal     97278
Name: count, dtype: int64


In [ ]:
protocol_type_distribution = df['protocol_type'].value_counts()
print(df['Attack_class'].value_counts())
print("Distribution of 'protocol_type' column:")
print(protocol_type_distribution)

Attack_class
attack    396743
normal     97278
Name: count, dtype: int64
Distribution of 'protocol_type' column:
protocol_type
icmp    283602
tcp     190065
udp      20354
Name: count, dtype: int64


In [ ]:
import numpy as np

df['Attack_class'] = np.where(df['label'] == 'normal.', 'normal', 'attack')
df.head()

print(df[['label', 'Attack_class']].head())

     label Attack_class
0  normal.       normal
1  normal.       normal
2  normal.       normal
3  normal.       normal
4  normal.       normal


In [ ]:
import numpy as np

df['Attack_class'] = np.where(df['label'] == 'normal.', 'normal', 'attack')

# Create the 'is_malicious_payload' column
df['is_malicious_payload'] = (df['Attack_class'] == 'attack')

print(df[['label', 'Attack_class', 'is_malicious_payload']].head())

     label Attack_class  is_malicious_payload
0  normal.       normal                 False
1  normal.       normal                 False
2  normal.       normal                 False
3  normal.       normal                 False
4  normal.       normal                 False


In [ ]:
pivot_table_data = df.pivot_table(index='protocol_type', columns='Attack_class', aggfunc='size', fill_value=0)
display(pivot_table_data)

Attack_class,attack,normal
protocol_type,,
icmp,282314,1288
tcp,113252,76813
udp,1177,19177


In [ ]:
service_pivot_table = df.pivot_table(index='service', columns='Attack_class', aggfunc='size', fill_value=0)

# Calculate total for each service
service_pivot_table['total'] = service_pivot_table['attack'] + service_pivot_table['normal']

# Calculate probabilities
service_pivot_table['probability_of_attack'] = service_pivot_table['attack'] / service_pivot_table['total']
service_pivot_table['probability_of_normal'] = service_pivot_table['normal'] / service_pivot_table['total']

# Rename columns for clarity
service_pivot_table = service_pivot_table.rename(columns={'probability_of_attack': 'attack', 'probability_of_normal': 'normal'})

print("Probabilities of attack and normal for each service type:")
display(service_pivot_table[['attack', 'normal']])

Probabilities of attack and normal for each service type:


Attack_class,attack,attack,normal,normal
service,,,,
IRC,1,0.023256,42,0.976744
X11,2,0.181818,9,0.818182
Z39_50,92,1.000000,0,0.000000
auth,108,0.329268,220,0.670732
bgp,106,1.000000,0,0.000000
...,...,...,...,...
urp_i,1,0.001859,537,0.998141
uucp,106,1.000000,0,0.000000
uucp_path,106,1.000000,0,0.000000


In [ ]:
pivot_table_data = pivot_table_data.rename(columns={'probability_of_attack': 'attack', 'probability_of_normal': 'normal'})
print("Pivot table with renamed probability columns:")
display(pivot_table_data[['attack', 'normal']])

Pivot table with renamed probability columns:


Attack_class,attack,normal
protocol_type,,
icmp,282314,1288
tcp,113252,76813
udp,1177,19177


In [ ]:
if 'total' not in pivot_table_data.columns:
    pivot_table_data['total'] = pivot_table_data['attack'] + pivot_table_data['normal']
if 'probability_of_attack' not in pivot_table_data.columns:
    pivot_table_data['probability_of_attack'] = pivot_table_data['attack'] / pivot_table_data['total']

pivot_table_data['probability_of_normal'] = pivot_table_data['normal'] / pivot_table_data['total']

print("Probabilities of attack and normal for each protocol type:")
display(pivot_table_data[['probability_of_attack', 'probability_of_normal']])

Probabilities of attack and normal for each protocol type:


Attack_class,probability_of_attack,probability_of_normal
protocol_type,,
icmp,0.995458,0.004542
tcp,0.595859,0.404141
udp,0.057826,0.942174


In [ ]:
pivot_table_data['total'] = pivot_table_data['attack'] + pivot_table_data['normal']
pivot_table_data['probability_of_attack'] = pivot_table_data['attack'] / pivot_table_data['total']

print("Probability of attack for each protocol type:")
display(pivot_table_data[['probability_of_attack']])